In [12]:
!pip install s3fs

  Using cached wrapt-1.16.0-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.6 kB)
  Using cached jmespath-1.0.1-py3-none-any.whl.metadata (7.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 kB 1.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 9.5 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 9.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.3/272.3 kB 6.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 128.7/128.7 kB 5.8 MB/s eta 0:00:00
Using cached wrapt-1.16.0-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl (80 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.1/328.1 kB 8.6 MB/s eta 0:00:00:00:01
Using cached jmespath-1.0.1-py3-none-any.whl (20 kB)
  Attempting

In [1]:
import cupy, cupyx
import cudf
import os
from datetime import datetime, timedelta
import os
from dotenv import load_dotenv

In [2]:
dotenv_path = 'aws_creds.txt'
load_dotenv(dotenv_path)

AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.environ.get("AWS_SESSION_TOKEN")

Sample CDR:

```
MSIDN:       IMSI:           IMEI:        PLAN:CALL_TYPE:CORRESP_TYPE:CORRESP_ISDN:DURATION:TIME:    DATE 
068373748102;208100167682477;351905149071;PLAN1;MOC;     CUST1;       0612287077;  247;     12:07:12;01/01/2012 
068373748102;208100167682477;351905149071;PLAN1;MTC;CUST2;0600000001;300;12:15:09;01/01/2012 
068373748102;208100167682477;351905149071;PLAN1;SMS-MO;CUST1;0613637193;0;12:18:18;01/01/2012 
068373748102;208100167682477;351905149071;PLAN1;SMS-MT;CUST1;0612899062;0;12:21:07;01/01/2012 
```

Sample network handover data:

```
CALL_ID:TOWER:EXCHANGE:DURATION:TIMESTAMP
0      ;1    ;1       ;233     ;01/01/2012T12:07:02
1      ;3    ;2       ;21      ;01/01/2012T12:07:05
1      ;2    ;3       ;113     ;01/01/2012T12:07:26
```

# CONFIGURATION

In [3]:
### save location
# save_folder = '../data'
save_folder = 's3://voltrondata-demo-data-us-east-2/telco/data/csv'

### customers
no_of_customers = 500000
avg_daily_record_number = no_of_customers * 500
no_of_contacts_range = (1, 25)

# this distribution roughly follows this chart: 
# https://www.researchgate.net/profile/Mario-Cools-2/publication/262526436/figure/fig3/AS:639170573762567@1529401548980/The-distribution-of-the-time-of-calls.png
call_time_distribution = cupy.array(
    [0.00876339, 0.00194742, 0.00097371, 0.00097371, 0.00194742,
       0.00681597, 0.01850049, 0.0399221 , 0.07302824, 0.0788705 ,
       0.07789679, 0.07400195, 0.06718598, 0.05744888, 0.05647517,
       0.05550146, 0.05744888, 0.05647517, 0.06621227, 0.0691334 ,
       0.05744888, 0.0399221 , 0.02239533, 0.01071081]
)
calls_share = .8
call_duration_poisson_lambda = 4
call_drop_rate = 0.012

call_types = cudf.DataFrame({
    'CALL_INDICATOR': [0,0,1,1],
    'CALL_DIRECTION': [0,1,0,1],
    'EVENT_TYPE': ['SMS-MO','SMS-MT','MOC','MTC']
})

# TOWERS
no_of_towers = 200
no_of_exchanges = 5

max_number_of_hops = 5
time_distribution_of_stationary_calls = cupy.array(
    [0.00068027, 0.00034014, 0.00034014, 0.00136054, 0.00306122,
       0.00408163, 0.0047619 , 0.00544218, 0.00612245, 0.00646259,
       0.00612245, 0.00510204, 0.0047619 , 0.00510204, 0.00544218,
       0.0047619 , 0.00442177, 0.00612245, 0.00646259, 0.00680272,
       0.00578231, 0.0037415 , 0.00170068, 0.00102041]    
)

In [6]:
### BRAZILIAN TELCO
# https://en.wikipedia.org/wiki/Mobile_network_codes_in_ITU_region_7xx_(South_America)#Brazil_–_BR
MCC = '724' # Brazil
MNCs = [f'{e:02d}' for e in [2,3,4,5,6,10,11,13,15,17,18,21,23,28,29,32,33,34,38,40,99]]

# https://en.wikipedia.org/wiki/List_of_dialling_codes_in_Brazil
AREA_CODES = cupy.array([
    11,12,13,14,15,16,17,18,19,21,22,24,27,28,31,32,33,34,35,37,38,41,42,43,44,45,46,47,48,49,
    51,53,54,55,61,62,64,63,65,66,67,68,69,71,73,74,75,76,77,78,79,81,87,82,83,84,85,88,86,89,
    91,93,94,92,97,95,96,98,99
])
START = 11111111
PLANS = [f'PLAN{e:02d}' for e in range(20)]

# GENERATE PHONE BOOK

In [7]:
max_contacts = 10
col = cupy.asarray(cupy.random.uniform(low=0, high=no_of_customers, size=(no_of_customers, max_contacts)), dtype='i').flatten()
row = cupy.asarray(cupy.random.uniform(low=0, high=no_of_customers, size=(no_of_customers, max_contacts)), dtype='i').flatten()
data = cupy.ones(row.size)
A = cupyx.scipy.sparse.coo_matrix((data, (row, col)), shape=(no_of_customers, no_of_customers))

In [8]:
del col, row, data
A_sym = (A + A.T)

The below is necessary as the `.todense()` call blows up the GPU memory.

In [9]:
stride = 500
contacts = cupy.empty((0,2), dtype='i')

for i in range(0, no_of_customers, stride):
    if i % 10000 == 0:
        print(i)
    temp = cupy.argwhere(A_sym[i:i+stride].todense() > 0)
    correct_idx = cupy.array([cupy.ones(temp.shape[0]) * i, cupy.zeros(temp.shape[0])], dtype='i').T
    temp = cupy.add(temp, correct_idx)
    contacts = cupy.append(contacts, temp, axis=0)

0
10000
20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000
170000
180000
190000
200000
210000
220000
230000
240000
250000
260000
270000
280000
290000
300000
310000
320000
330000
340000
350000
360000
370000
380000
390000
400000
410000
420000
430000
440000
450000
460000
470000
480000
490000


In [7]:
contacts.size

19999544

In [13]:
cudf.DataFrame(contacts, columns=['FROM', 'TO']).to_csv(save_folder + '/temp/contacts.csv', index=False, storage_options={
        "key": AWS_ACCESS_KEY_ID,
        "secret": AWS_SECRET_ACCESS_KEY,
        "token": AWS_SESSION_TOKEN,
    })

In [6]:
contacts_df = cudf.read_csv(save_folder + '/temp/contacts.csv')

In [13]:
contacts = contacts_df.to_cupy()

# GENERATE TOWERS

In [15]:
towers = cupy.reshape(cupy.array([range(no_of_towers)]), (no_of_towers, 1))
exchanges = cupy.reshape(cupy.asarray(cupy.random.uniform(0, no_of_exchanges, size=no_of_towers), dtype='i'), (no_of_towers,1))
towers = cupy.append(towers, exchanges, axis=1)

In [16]:
cudf.DataFrame(towers, columns=['TOWER_ID', 'EXACHANGE']).to_csv(save_folder + '/temp/towers.csv', index=False, storage_options={
        "key": AWS_ACCESS_KEY_ID,
        "secret": AWS_SECRET_ACCESS_KEY,
        "token": AWS_SESSION_TOKEN,
    })

In [15]:
towers_df = cudf.read_csv(save_folder + '/temp/towers.csv')
towers = towers_df.to_cupy()

# GENERATE MSIDN, IMSI, IMEI, AND PLAN

In [17]:
area_codes = cupy.asarray(cupy.random.uniform(0, len(AREA_CODES), (no_of_customers, 1)), dtype='i')
msidns = cupy.array([[START + e,] for e in range(no_of_customers)])

customer_metadata_temp = cupy.append(cupy.array([[724 * 10e10,]] * no_of_customers), AREA_CODES[area_codes] * 10e8, axis=1)
customer_metadata_temp = cupy.append(customer_metadata_temp, msidns, axis=1)
CUSTOMER_ID = cupy.array(range(0, no_of_customers))
IMSI = cupy.asarray(cupy.matmul(customer_metadata_temp, cupy.array([[[1],[1],[1]]])), dtype='int64')
MSIDN = cupy.asarray(cupy.matmul(customer_metadata_temp[:,[1,2]], cupy.array([[[1],[1]]])), dtype='int64')
IMEI = cupy.asarray(cupy.random.uniform(0, 999999999999999, size=no_of_customers), dtype='int64')
PLAN = cupy.asarray(cupy.random.uniform(0, 20, size=no_of_customers), dtype='i')

customer_metadata = cupy.append(cupy.reshape(CUSTOMER_ID, (CUSTOMER_ID.shape[0],1)), cupy.reshape(MSIDN, (MSIDN.shape[1],1)), axis=1)
customer_metadata = cupy.append(customer_metadata, cupy.reshape(IMSI, (IMSI.shape[1],1)), axis=1)
customer_metadata = cupy.append(customer_metadata, cupy.reshape(IMEI, (IMEI.shape[0],1)), axis=1)
customer_metadata = cupy.append(customer_metadata, cupy.reshape(PLAN, (PLAN.shape[0],1)), axis=1)

customer_metadata = cudf.DataFrame(customer_metadata, columns=['CUSTOMER_ID','MSIDN','IMSI','IMEI','PLAN'])

In [18]:
customer_metadata.to_csv(save_folder + '/temp/customer_meta.csv', index=False, storage_options={
        "key": AWS_ACCESS_KEY_ID,
        "secret": AWS_SECRET_ACCESS_KEY,
        "token": AWS_SESSION_TOKEN,
    })

In [21]:
customer_metadata = cudf.read_csv(save_folder + '/temp/customer_meta.csv')
# customer_metadata = customer_metadata_df.to_cupy()

# MAIN LOOP

In [18]:
def check_folder(partition_id, call_or_network='cdr'):
    pth = f'{save_folder}/{call_or_network}/{partition_id}'
    if not os.path.isdir(pth):
        os.mkdir(pth)

def post_process_data(records, day, hour, quarter):
    temp = (
        records
        .merge(customer_metadata, left_on='CUSTOMER_ID', right_on='CUSTOMER_ID', how='left')
        .merge(customer_metadata[['CUSTOMER_ID','MSIDN']], left_on='CUSTOMER_CORRESPONDING', right_on='CUSTOMER_ID', how='left', suffixes=('', '_CORRESPONDING'))
        .merge(call_types, on=['CALL_INDICATOR', 'CALL_DIRECTION'], how='left')
    )
    
    temp['MSIDN'] = '0' + temp['MSIDN'].astype('str')
    temp['MSIDN_CORRESPONDING'] = '0' + temp['MSIDN_CORRESPONDING'].astype('str')
    temp['PLAN'] = 'PLAN' + temp['PLAN'].astype('str')

    ## TODO -- fix this to follow the code below on how we generate time
    temp['START_DATETIME'] = cudf.to_datetime('2020-01-01')
    temp['START_TIMEDELTA'] = (
        (temp['START_DAY'] * 24 + temp['START_HOUR']) * 3600 
        + temp['START_SECOND']).astype('timedelta64[s]')

    temp['START_DATETIME'] = temp['START_DATETIME'] + temp['START_TIMEDELTA']    
    temp['EVENT_START_TIME'] = temp['START_DATETIME'].dt.strftime('%H:%M:%S')
    temp['EVENT_START_DATE'] = temp['START_DATETIME'].dt.strftime('%Y/%m/%d')
    temp = temp[['RECORD_ID','MSIDN','IMSI','IMEI','PLAN','EVENT_TYPE','ERROR_INDICATOR','CALL_DURATION','MSIDN_CORRESPONDING','EVENT_START_TIME','EVENT_START_DATE']]

    # check if the folder exists and if not -- create it
    dt = (datetime(2020, 1,1) + timedelta(days=day)).strftime('%Y%m%d')
    # check_folder(dt)
    
    # save the data 
    temp.to_csv(save_folder + f'/cdr/{dt}/cdr_{dt}_{hour * 100 + quarter*15:04d}.csv', sep=';', index=False, chunksize=10000, storage_options={
        "key": AWS_ACCESS_KEY_ID,
        "secret": AWS_SECRET_ACCESS_KEY,
        "token": AWS_SESSION_TOKEN,
    })

def generate_network_data(records, no_of_records, call_or_text, day, hour,quarter):
    # figure out which calls were 'mobile'
    temp_records = cupy.random.uniform(0, 1, no_of_records) <= time_distribution_of_stationary_calls[0]
    mobile_records = cupy.argwhere(temp_records * call_or_text > 0)

    # figure out how many hops they made
    occurences = cupy.asarray(cupy.random.uniform(2, max_number_of_hops, mobile_records.shape[0]), dtype='i')

    # concatenate the repeat calls with the records
    repeats = cupy.repeat(mobile_records, occurences.tolist())
    repeats = records[repeats]

    # convert to a dataframe for ease of operating on the records
    network_records = cupy.append(cupy.delete(records, mobile_records.flatten(), axis=0), repeats, axis=0)[:,[0,3,5,7,8,9]]
    network_records = cudf.DataFrame(network_records, columns=['RECORD_ID','CALL_INDICATOR','CALL_DURATION','START_SECOND','START_HOUR', 'START_DAY'])
    network_records = network_records.sort_values(by='RECORD_ID').reset_index(drop=True)

    del records, temp_records, mobile_records

    # append tower information
    record_towers = towers[cupy.asarray(cupy.random.uniform(0,towers.shape[0],network_records.shape[0]), dtype='i')]
    record_tower_duration_rnd = cupy.random.uniform(0,towers.shape[0],network_records.shape[0])
    record_towers = cudf.DataFrame(
        cupy.append(record_towers, cupy.reshape(record_tower_duration_rnd, (network_records.shape[0], 1)), axis=1),
        columns=['TOWER_ID', 'EXCHANGE_ID', 'RND_DUR_SHARE']
    )

    # split the duration of call between towers
    record_towers['RECORD_ID'] = network_records['RECORD_ID']
    summary = record_towers.groupby(by='RECORD_ID').sum().reset_index()[['RECORD_ID','RND_DUR_SHARE']]
    record_towers = record_towers.merge(summary, on='RECORD_ID')
    record_towers['REC_DUR'] = record_towers['RND_DUR_SHARE_x'] / record_towers['RND_DUR_SHARE_y']
    record_towers = record_towers.sort_values(by='RECORD_ID').reset_index(drop=True)

    # calculate the event duration
    network_records = network_records.join(record_towers, lsuffix='_net', rsuffix='_tow')

    del record_towers
    
    network_records['EVENT_DURATION'] = (network_records['REC_DUR'] * network_records['CALL_DURATION']).astype('int')
    network_records = network_records.rename(columns={'RECORD_ID_net': 'RECORD_ID'}).reset_index(drop=True)

    # calculate the event date and time
    network_records['RECORD_DUR_CUMSUM'] = network_records.set_index('RECORD_ID')['EVENT_DURATION'].groupby('RECORD_ID').cumsum().reset_index(drop=True)
    network_records['START_SECOND_TOWER'] = network_records['START_SECOND'] + network_records['RECORD_DUR_CUMSUM']
    network_records['START_DATETIME'] = cudf.to_datetime('2020-01-01')
    network_records['START_TIMEDELTA'] = (
        (network_records['START_DAY'] * 24 + network_records['START_HOUR']) * 3600 
        + network_records['START_SECOND_TOWER']).astype('timedelta64[s]')

    network_records['START_DATETIME'] = network_records['START_DATETIME'] + network_records['START_TIMEDELTA']    
    network_records['EVENT_START_TIME'] = network_records['START_DATETIME'].dt.strftime('%H:%M:%S')
    network_records['EVENT_START_DATE'] = network_records['START_DATETIME'].dt.strftime('%Y/%m/%d')
    
    network_records = network_records.sort_values(by='RECORD_ID').reset_index(drop=True)
    network_records = network_records[['RECORD_ID','CALL_INDICATOR','TOWER_ID','EXCHANGE_ID','EVENT_DURATION','EVENT_START_TIME', 'EVENT_START_DATE']]

    # save the data
    dt = (datetime(2020, 1,1) + timedelta(days=day)).strftime('%Y%m%d')
    # check_folder(dt, 'ndr')

    network_records.to_csv(save_folder + f'/ndr/{dt}/ndr_{dt}_{hour * 100 + quarter*15:04d}.csv', sep=';', index=False, chunksize=10000, storage_options={
        "key": AWS_ACCESS_KEY_ID,
        "secret": AWS_SECRET_ACCESS_KEY,
        "token": AWS_SESSION_TOKEN,
    })

In [22]:
%%time
for day in range(0, 2):
    record_no = 0
    
    for hour in range(23, 24):
        print(f'Processing day: {day}, hour: {hour} ')
        
        # how many records to generate during this hour?
        no_of_records_hr = int(call_time_distribution[hour] * avg_daily_record_number)

        if no_of_records_hr == 0:
            continue

        # split into 15 minute-chunks
        quarter_hr_records = cupy.random.uniform(0, 100, 4)
        quarter_hr_records = cupy.asarray(quarter_hr_records / quarter_hr_records.sum() * no_of_records_hr, dtype='i')

        for quarter in range(0, 4):
            no_of_records = quarter_hr_records[quarter].tolist()

            # create record_id
            # it restarts every day
            records = cupy.reshape(cupy.asarray(cupy.cumsum(cupy.ones(no_of_records)) + record_no - 1, dtype='i'), (no_of_records, 1))
            record_no += no_of_records
            
            # select contacts who will be included
            contacts = contacts[cupy.asarray(cupy.random.uniform(0, contacts.size, no_of_records), dtype='i')]
            
            # phone call or text?
            call_or_text = cupy.asarray(cupy.random.uniform(0, 1, no_of_records) < calls_share, dtype='i')
            call_drop = call_or_text * cupy.asarray(cupy.random.uniform(0, 1, no_of_records) < call_drop_rate, dtype='i')
            call_duration = cupy.asarray(cupy.random.poisson(lam=4, size=no_of_records) * cupy.random.uniform(10, 200, no_of_records) + 1, dtype='i') * call_or_text
            call_direction = cupy.asarray(cupy.random.uniform(-1, 1, no_of_records) >= 0, dtype='i')
            
            # generate seconds for this hour
            seconds_from, seconds_to = quarter * 900, (quarter + 1) * 900
            time_of_call = cupy.asarray(cupy.random.uniform(seconds_from, seconds_to, no_of_records), dtype='i')
    
            # concatenate all the data
            records = cupy.append(records, contacts, axis=1)
            records = cupy.append(records, cupy.reshape(call_or_text, (no_of_records, 1)), axis=1)
            records = cupy.append(records, cupy.reshape(call_drop, (no_of_records, 1)), axis=1)
            records = cupy.append(records, cupy.reshape(call_duration, (no_of_records, 1)), axis=1)
            records = cupy.append(records, cupy.reshape(call_direction, (no_of_records, 1)), axis=1)
            records = cupy.append(records, cupy.reshape(time_of_call, (no_of_records, 1)), axis=1)
            records = cupy.append(records, cupy.array([[hour]] * no_of_records), axis=1)
            records = cupy.append(records, cupy.array([[day]] * no_of_records), axis=1)
            
            # move to cudf and post-process
            print(f'\tQuarter hour: {quarter}. ', end="", flush=True)
            records_df = cudf.DataFrame(records, columns=['RECORD_ID','CUSTOMER_ID','CUSTOMER_CORRESPONDING','CALL_INDICATOR','ERROR_INDICATOR','CALL_DURATION','CALL_DIRECTION','START_SECOND','START_HOUR', 'START_DAY'])
            post_process_data(records_df, day, hour, quarter)
            print('\b\tGenerated CDR data... ', end="", flush=True)
            generate_network_data(records, no_of_records, call_or_text, day, hour, quarter)
            print('Generated NDR data...')
        

Processing day: 0, hour: 23 
	Quarter hour: 0.	Generated CDR data... Generated NDR data...
	Quarter hour: 1.	Generated CDR data... Generated NDR data...
	Quarter hour: 2.	Generated CDR data... Generated NDR data...
	Quarter hour: 3.	Generated CDR data... Generated NDR data...
Processing day: 1, hour: 23 
	Quarter hour: 0.	Generated CDR data... Generated NDR data...
	Quarter hour: 1.	Generated CDR data... Generated NDR data...
	Quarter hour: 2.	Generated CDR data... Generated NDR data...
	Quarter hour: 3.	Generated CDR data... Generated NDR data...
CPU times: user 7.98 s, sys: 1.1 s, total: 9.09 s
Wall time: 3min 20s
